### 한국어 형태소 분석
- kiwi
- 한국어는 영어와 달라서, 조사와 어미가 덕지덕지 붙어 있음 (표현력이 남다르긴 하다)
- 영어는 단어끼리 띄어쓰기 구분되고 복수형 s 등 정형화돼있지만 한국어는 그렇지 않음
- 형태소 분석을 통해서 내용들을 확인할 필요가 있다.
- TF-IDF 빈도 분석도 진행

In [2]:
import pandas as pd

df = pd.read_csv("../data/11-1_뉴스정제.csv")
df[['제목', '정제본문','카테고리']].head(3)

,제목,정제본문,카테고리
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...,경제
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...,경제


In [5]:
from kiwipiepy import Kiwi

kiwi = Kiwi()
print(kiwi)

Kiwi(num_workers=16, model_path=None, integrate_allomorph=True, load_default_dict=True, load_typo_dict=True, model_type='cong', enabled_dialects=<Dialect.STANDARD: 0>)


In [10]:
#  경우 1 - 문장을 형태소로 나누기

sentence = "오늘은 날씨가 정말 좋아서 소풍을 꼭 가보고 싶다. 이 곳에 가면 좋을 것 같다."
result = kiwi.tokenize(sentence)
result

[Token(form='오늘', tag='NNG', start=0, len=2),
 Token(form='은', tag='JX', start=2, len=1),
 Token(form='날씨', tag='NNG', start=4, len=2),
 Token(form='가', tag='JKS', start=6, len=1),
 Token(form='정말', tag='MAG', start=8, len=2),
 Token(form='좋', tag='VA', start=11, len=1),
 Token(form='어서', tag='EC', start=12, len=2),
 Token(form='소풍', tag='NNG', start=15, len=2),
 Token(form='을', tag='JKO', start=17, len=1),
 Token(form='꼭', tag='MAG', start=19, len=1),
 Token(form='가', tag='VV', start=21, len=1),
 Token(form='어', tag='EC', start=21, len=1),
 Token(form='보', tag='VX', start=22, len=1),
 Token(form='고', tag='EC', start=23, len=1),
 Token(form='싶', tag='VX', start=25, len=1),
 Token(form='다', tag='EF', start=26, len=1),
 Token(form='.', tag='SF', start=27, len=1),
 Token(form='이', tag='MM', start=29, len=1),
 Token(form='곳', tag='NNG', start=31, len=1),
 Token(form='에', tag='JKB', start=32, len=1),
 Token(form='가', tag='VV', start=34, len=1),
 Token(form='면', tag='EC', start=35, len=1),
 

In [11]:
for token in result:
    print(token.form, token.tag)    # form은 실제 단어, tag는 형태소 태그명

오늘 NNG
은 JX
날씨 NNG
가 JKS
정말 MAG
좋 VA
어서 EC
소풍 NNG
을 JKO
꼭 MAG
가 VV
어 EC
보 VX
고 EC
싶 VX
다 EF
. SF
이 MM
곳 NNG
에 JKB
가 VV
면 EC
좋 VA
을 ETM
것 NNB
같 VA
다 EF
. SF


In [ ]:
# 명사만 뽑아보기

nouns = []

for token in result:
    if token.tag.startswith("N"):   # 명사는 N으로 시작하므로!
        nouns.append(token.form)

nouns   # 뒤에 곳이랑 것은 제거하는 게 좋다

['오늘', '날씨', '소풍', '곳', '것']

In [13]:
# 동사만 뽑아보기

verbs = []

for token in result:
    if token.tag.startswith("V"):   # 명사는 N으로 시작하므로!
        verbs.append(token.form)

verbs

['좋', '가', '보', '싶', '가', '좋', '같']

In [14]:
# 한 글자 명사 걸러내기
nouns_filtered = []

for noun in nouns:
    if len(noun) > 1:
        nouns_filtered.append(noun)

nouns_filtered

['오늘', '날씨', '소풍']

In [15]:
df[['제목', '정제본문']]

,제목,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...
3,오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은,img tag s 지난 30일 서울의 한 주유소 사진 연합뉴스 img tag e 오...
4,푸르덴셜생명 더 큰 드림 변액연금보험Ⅱ에 신규펀드 13종 추가,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...
...,...,...
995,누리호 KAIST팀 양방향 교신 성공..임무 수행도 기대,위성 전력 충전 상태 안정적으로 작동 지상국 명령 따라 정상 임무 확인 위성 안정적...
996,로봇이 된 안마의자…바디프랜드 600만원대 ‘팬텀 로보’ 출시,R D 비용 50억원 필라테스 사이클 동작 구현 가능 연내 생체신호 센서 탑재 제품...
997,직방 박영걸 CTO 직방의 홈 IoT 사업은 더 살기 좋은 집 위한 기술,직방이 궁극적으로 추구하는 방향은 라이프스타일 플랫폼이다 초창기에는 원룸 위주의 부...
998,스타트업 돋보기네이버카카오가 점찍은 플로틱 왜 이커머스 물류센터 주목할까,요즘 스타트업의 비즈니스 모델 을 살펴봅니다 네이버와 카카오가 보여주는 스타트업 투...


In [18]:
# 명사 추출 함수 만들기

def extract_nouns(text):
    nouns = []
    result = kiwi.tokenize(text)

    for token in result:
        if token.tag.startswith("N"):
            nouns.append(token.form)

    nouns_filtered = []

    for noun in nouns:
        if len(noun) > 1:
            nouns_filtered.append(noun)

    return nouns_filtered

In [22]:
# 데이터프레임에 전체 적용

df['명사'] = df['정제본문'].apply(extract_nouns)
df[['정제본문', '명사']].head(3)

,정제본문,명사
0,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,"[서울, 연합뉴스, 현대백화점그룹, 광주광역시, 서울, 여의도, 현대, 서울, 문화..."
1,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...,"[전주, 뉴시스, 김얼, 기자, 이스타항공, 자금, 배임, 횡령, 전주, 교도소, ..."
2,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...,"[NH농협은행, 올해, 농협, 금융, 출범, 주년, 이달, 주년, 기념, 주화, 대..."
